In [1]:
# ==============================================================================
# 11. Model Preparation (모델 학습 준비)
# ==============================================================================
#
# 목적:
#   1. Train/Val/Test 분할 (환자 단위 - Data Leakage 방지)
#   2. 피처/레이블 분리
#   3. 클래스 불균형 확인
#
# 입력: features_final.csv
# 출력: train.csv, val.csv, test.csv
# ==============================================================================

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
import json

INPUT_DIR = '../data/processed'

# 전체 자동 추출 아닐 경우 주석 처리
# OUTPUT_DIR = '../data/processed'

# 전체 자동 추출 말고 feature 수 제한할 경우 여기만 바꾸면 됩니다 (e.g. top33, top21, top10, top5)
FEATURE_CONFIG = 'top21'
with open(os.path.join(INPUT_DIR, f'{FEATURE_CONFIG}_features.json'), 'r') as f:
    feature_cols = json.load(f)
OUTPUT_DIR = f'../data/processed/{FEATURE_CONFIG}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=== 11. Model Preparation 시작 ===")

# --- 데이터 로드 ---
print("\nStep 1: 데이터 로드")

df = pd.read_csv(os.path.join(INPUT_DIR, 'features_final.csv'))
print(f"✓ 데이터 로드 완료: {len(df):,} rows")
print(f"  - 고유 환자 수: {df['stay_id'].nunique():,}명")
print(f"  - 컬럼 수: {len(df.columns)}개")
print(f"  - 피처 로드: {FEATURE_CONFIG} ({len(feature_cols)}개)")

=== 11. Model Preparation 시작 ===

Step 1: 데이터 로드
✓ 데이터 로드 완료: 941,817 rows
  - 고유 환자 수: 23,938명
  - 컬럼 수: 84개
  - 피처 로드: top21 (21개)


In [2]:
# ==============================================================================
# Step 2: 피처/레이블 정의
# ==============================================================================

print("\nStep 2: 피처/레이블 정의")

# --- ID 컬럼 ---
id_cols = ['stay_id', 'subject_id', 'hadm_id', 'observation_hour', 'observation_start', 'observation_end']

# --- 레이블 컬럼 ---
label_cols = [col for col in df.columns if 'next_' in col]

# --- 피처 컬럼 ---
# 자동 추출 (모든 feature 지정)
# feature_cols = [col for col in df.columns if col not in id_cols + label_cols]
# 수동 지정 (e.g. top33, top21, etc)
# feature_cols = top21_features

print(f"  - ID 컬럼: {len(id_cols)}개")
print(f"  - 피처 컬럼: {len(feature_cols)}개")
print(f"  - 레이블 컬럼: {len(label_cols)}개")

# --- 레이블 목록 ---
print(f"\n=== 레이블 목록 ===")
for col in label_cols:
    pos_rate = df[col].mean() * 100
    print(f"  {col}: {pos_rate:.2f}% positive")


Step 2: 피처/레이블 정의
  - ID 컬럼: 6개
  - 피처 컬럼: 21개
  - 레이블 컬럼: 12개

=== 레이블 목록 ===
  death_next_6h: 0.19% positive
  vent_next_6h: 0.96% positive
  pressor_next_6h: 0.48% positive
  composite_next_6h: 1.39% positive
  death_next_12h: 0.40% positive
  vent_next_12h: 1.64% positive
  pressor_next_12h: 0.86% positive
  composite_next_12h: 2.43% positive
  death_next_24h: 0.92% positive
  vent_next_24h: 2.58% positive
  pressor_next_24h: 1.41% positive
  composite_next_24h: 4.01% positive


In [3]:
# ==============================================================================
# Step 3: Train/Val/Test 분할 (환자 단위)
# ==============================================================================
#
# ⚠️ 중요: 행 단위가 아닌 "환자 단위"로 분할해야 함
#
# 이유 (Data Leakage 방지):
#   - 같은 환자가 train과 test에 동시에 있으면 cheating
#   - 환자 A의 6h 데이터로 학습 → 환자 A의 12h 예측 → 당연히 잘 맞음
#   - 실제 배포 시에는 "처음 보는 환자"를 예측해야 함
#
# 분할 비율: Train 70% / Val 15% / Test 15%
# ==============================================================================

print("\nStep 3: Train/Val/Test 분할 (환자 단위)")

# --- 고유 환자 추출 ---
unique_patients = df['stay_id'].unique()
print(f"  총 환자 수: {len(unique_patients):,}명")

# --- 1차 분할: Train (70%) vs Temp (30%) ---
train_patients, temp_patients = train_test_split(
    unique_patients,
    test_size=0.30,
    random_state=42
)

# --- 2차 분할: Val (15%) vs Test (15%) ---
val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=42
)

print(f"\n  환자 수 분할:")
print(f"    - Train: {len(train_patients):,}명 ({len(train_patients)/len(unique_patients)*100:.1f}%)")
print(f"    - Val: {len(val_patients):,}명 ({len(val_patients)/len(unique_patients)*100:.1f}%)")
print(f"    - Test: {len(test_patients):,}명 ({len(test_patients)/len(unique_patients)*100:.1f}%)")

# --- DataFrame 분할 ---
df_train = df[df['stay_id'].isin(train_patients)].copy()
df_val = df[df['stay_id'].isin(val_patients)].copy()
df_test = df[df['stay_id'].isin(test_patients)].copy()

print(f"\n  행 수 분할:")
print(f"    - Train: {len(df_train):,} rows ({len(df_train)/len(df)*100:.1f}%)")
print(f"    - Val: {len(df_val):,} rows ({len(df_val)/len(df)*100:.1f}%)")
print(f"    - Test: {len(df_test):,} rows ({len(df_test)/len(df)*100:.1f}%)")


Step 3: Train/Val/Test 분할 (환자 단위)
  총 환자 수: 23,938명

  환자 수 분할:
    - Train: 16,756명 (70.0%)
    - Val: 3,591명 (15.0%)
    - Test: 3,591명 (15.0%)

  행 수 분할:
    - Train: 657,172 rows (69.8%)
    - Val: 141,844 rows (15.1%)
    - Test: 142,801 rows (15.2%)


In [4]:
# ==============================================================================
# Step 4: 클래스 불균형 확인
# ==============================================================================
#
# 예상: death_next_6h ~ 0.19%, composite_next_24h ~ 4%
# 불균형 심하면 class_weight 또는 scale_pos_weight 필요
# ==============================================================================

print("\nStep 4: 클래스 불균형 확인")

print("\n=== Train 세트 레이블 분포 ===")
for col in label_cols:
    pos_count = df_train[col].sum()
    neg_count = len(df_train) - pos_count
    pos_rate = df_train[col].mean() * 100
    imbalance_ratio = neg_count / pos_count if pos_count > 0 else float('inf')
    
    # 심각도 표시
    if pos_rate < 1:
        severity = "🔴 매우 심각"
    elif pos_rate < 3:
        severity = "🟡 심각"
    else:
        severity = "🟢 중간"
    
    print(f"  {col}:")
    print(f"    - Positive: {pos_count:,} ({pos_rate:.2f}%)")
    print(f"    - Negative: {neg_count:,}")
    print(f"    - Imbalance Ratio: 1:{imbalance_ratio:.0f} {severity}")


Step 4: 클래스 불균형 확인

=== Train 세트 레이블 분포 ===
  death_next_6h:
    - Positive: 1,296 (0.20%)
    - Negative: 655,876
    - Imbalance Ratio: 1:506 🔴 매우 심각
  vent_next_6h:
    - Positive: 6,307 (0.96%)
    - Negative: 650,865
    - Imbalance Ratio: 1:103 🔴 매우 심각
  pressor_next_6h:
    - Positive: 3,290 (0.50%)
    - Negative: 653,882
    - Imbalance Ratio: 1:199 🔴 매우 심각
  composite_next_6h:
    - Positive: 9,343 (1.42%)
    - Negative: 647,829
    - Imbalance Ratio: 1:69 🟡 심각
  death_next_12h:
    - Positive: 2,803 (0.43%)
    - Negative: 654,369
    - Imbalance Ratio: 1:233 🔴 매우 심각
  vent_next_12h:
    - Positive: 10,827 (1.65%)
    - Negative: 646,345
    - Imbalance Ratio: 1:60 🟡 심각
  pressor_next_12h:
    - Positive: 5,889 (0.90%)
    - Negative: 651,283
    - Imbalance Ratio: 1:111 🔴 매우 심각
  composite_next_12h:
    - Positive: 16,326 (2.48%)
    - Negative: 640,846
    - Imbalance Ratio: 1:39 🟡 심각
  death_next_24h:
    - Positive: 6,366 (0.97%)
    - Negative: 650,806
    - Imbalance

In [5]:
# ==============================================================================
# Step 5: scale_pos_weight 계산 (XGBoost/LightGBM용)
# ==============================================================================
#
# scale_pos_weight = negative_count / positive_count
# 모델 학습 시 positive 클래스에 가중치 부여
# ==============================================================================

print("\nStep 5: scale_pos_weight 계산")

scale_pos_weights = {}

for col in label_cols:
    pos_count = df_train[col].sum()
    neg_count = len(df_train) - pos_count
    
    if pos_count > 0:
        scale_pos_weights[col] = neg_count / pos_count
    else:
        scale_pos_weights[col] = 1.0

print("\n=== scale_pos_weight (XGBoost/LightGBM용) ===")
for col, weight in scale_pos_weights.items():
    print(f"  {col}: {weight:.1f}")

# 저장 (모델 학습 시 사용)
import json
weights_path = os.path.join(OUTPUT_DIR, 'scale_pos_weights.json')
with open(weights_path, 'w') as f:
    json.dump(scale_pos_weights, f, indent=2)
print(f"\n✓ scale_pos_weight 저장: {weights_path}")


Step 5: scale_pos_weight 계산

=== scale_pos_weight (XGBoost/LightGBM용) ===
  death_next_6h: 506.1
  vent_next_6h: 103.2
  pressor_next_6h: 198.7
  composite_next_6h: 69.3
  death_next_12h: 233.5
  vent_next_12h: 59.7
  pressor_next_12h: 110.6
  composite_next_12h: 39.3
  death_next_24h: 102.2
  vent_next_24h: 37.6
  pressor_next_24h: 68.0
  composite_next_24h: 23.4

✓ scale_pos_weight 저장: ../data/processed/top21/scale_pos_weights.json


In [6]:
# ==============================================================================
# Step 6: 저장
# ==============================================================================

print("\n" + "="*60)
print("Step 6: 저장")
print("="*60)

# --- CSV 저장 ---
train_path = os.path.join(OUTPUT_DIR, 'train.csv')
val_path = os.path.join(OUTPUT_DIR, 'val.csv')
test_path = os.path.join(OUTPUT_DIR, 'test.csv')

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path, index=False)
df_test.to_csv(test_path, index=False)

print(f"\n✓ 저장 완료:")
print(f"  - train.csv: {len(df_train):,} rows, {os.path.getsize(train_path)/1024/1024:.1f} MB")
print(f"  - val.csv: {len(df_val):,} rows, {os.path.getsize(val_path)/1024/1024:.1f} MB")
print(f"  - test.csv: {len(df_test):,} rows, {os.path.getsize(test_path)/1024/1024:.1f} MB")

# --- 피처 목록 저장 ---
feature_list_path = os.path.join(OUTPUT_DIR, 'feature_cols.json')
with open(feature_list_path, 'w') as f:
    json.dump(feature_cols, f, indent=2)
print(f"  - feature_cols.json: {len(feature_cols)}개 피처")

print("\n=== 11. Model Preparation 완료 ===")


Step 6: 저장

✓ 저장 완료:
  - train.csv: 657,172 rows, 522.7 MB
  - val.csv: 141,844 rows, 112.9 MB
  - test.csv: 142,801 rows, 113.6 MB
  - feature_cols.json: 21개 피처

=== 11. Model Preparation 완료 ===
